# RAG con Ollama y FAISS: gobernanza de datos

Este notebook implementa un pipeline de RAG sobre el documento `rag_pdf_files/Gobernanza_Datos.pdf`.

- **Embeddings:** `nomic-embed-text` (768 dimensiones).
- **Generación:** `llama3.2` con temperatura 0.
- **Fragmentación:** 1000 caracteres por chunk y 200 de solape.
- **Recuperación:** los 3 chunks más similares (`k=3`) se usan como contexto.


In [ ]:
#Ollama dependencies
!sudo apt update
!sudo apt-get install -y zstd
!sudo apt install -y pciutils
!sudo apt-get install -y poppler-utils libmagic-dev
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
%pip install pypdf
%pip install -U langchain langchain-text-splitters langchain-ollama langchain-unstructured langchain-community unstructured
%pip install faiss-cpu
%pip install unstructured[pdf]

In [ ]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [ ]:
!ollama pull llama3.2

In [ ]:
#Large language models (like llama3.1) and embedding models (like nomic-embed-text) must be downloaded to your machine separately before your Python scripts can access them
!ollama pull nomic-embed-text

In [ ]:
import os
import glob
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
# Modern Ollama Imports
from langchain_ollama import ChatOllama, OllamaEmbeddings
# Modern Directory Loader Import
# from langchain_unstructured import
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
import html
from IPython.display import display, HTML

# --- 1. Load Documents ---
def load_documents_from_directory(dir_path: str):
    file_paths = sorted(glob.glob(f"{dir_path}/*.pdf*"))

    if not file_paths:
        print(f"No files found in {dir_path}")
        return []

    print(f"Loading following files: {file_paths}")

    documents = []
    for file_path in file_paths:
        loader = PyPDFLoader(file_path)
        documents.extend(loader.load())

    print(f"Páginas cargadas: {len(documents)}")
    return documents

# --- 2. Build or Load Vector Database ---
def create_vector_store(documents, embedding_model_name="nomic-embed-text"):
    # Initialize the modern Ollama embedding class
    embeddings = OllamaEmbeddings(model=embedding_model_name)

    # Split text into manageable chunks
    text_splitter = RecursiveCharacterTextSplitter(
        # 1000 caracteres (~250-300 tokens) conserva una idea completa sin diluir el embedding.
        # 200 de solape (20 %) evita cortar definiciones o frases en los bordes.
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len
    )
    chunks = text_splitter.split_documents(documents)

    print(f"Total de chunks generados: {len(chunks)}")
    print(f"Longitud promedio: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} caracteres")

    # Generate embeddings and populate FAISS database
    print(f"Generating embeddings for {len(chunks)} text chunks...")

    vector_store = FAISS.from_documents(chunks, embeddings)
    return vector_store

def format_docs(docs):
    formatted_docs = "\n\n".join(doc.page_content for doc in docs)

    print(f"\n------Retrieve docs:")
    for doc in docs:
      print(doc.page_content)
      print("\n")
    print("\n To be used as context for the RAG query.\n")

    return formatted_docs

# --- 3. Execute RAG Query Loop ---
def run_rag_pipeline(vector_store, model_name="llama3.2"):
    # Instantiate the modern ChatOllama class for generation
    llm = ChatOllama(model=model_name, temperature=0)

    # Define a retriever from our FAISS index
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})

    while True:
        query = input("\nAsk a question about your documents (or type 'exit'): ")
        if query.lower() == 'exit':
            break

        # Retrieve relevant text context
        #relevant_docs = retriever.invoke(query)
        #format_docs = "\n\n".join([doc.page_content for doc in relevant_docs])

        #Ajusta el prompt para que el modelo genere respuestas basadas en el contexto recuperado
        prompt_instance = ChatPromptTemplate.from_template("""
            Eres un asistente que responde preguntas basándose únicamente en el contexto proporcionado.
            Si no sabes la respuesta, dilo explícitamente. No inventes información.

            Contexto:
            {context}

            Pregunta: {question}
            Respuesta:
            """)

        rag_chain = (
                {
                    "context": itemgetter("question") | retriever | format_docs,
                    "question": itemgetter("question")
                }
                | prompt_instance
                | llm
                | StrOutputParser()
        )

        # Generate and stream the response from Ollama
        answer = rag_chain.invoke({"question": query})

        print("\n--- Answer ---")
        display(HTML(
            "<div style='white-space: pre-wrap; overflow-wrap: anywhere; "
            "line-height: 1.5; max-width: 100%;'>"
            + html.escape(answer)
            + "</div>"
        ))
        print("\n--------------")

In [ ]:
import os

# Crea el directorio en colab si no existe, subir el documento ahí
os.makedirs("/content/rag_pdf_files", exist_ok=True)
print("Carpeta creada:", os.path.isdir("/content/rag_pdf_files"))
print(os.listdir("/content"))

In [ ]:
# --- Main Execution Execution ---
if __name__ == "__main__":
    DOCS_DIR = "rag_pdf_files"

    # 1. Ingest
    raw_docs = load_documents_from_directory(DOCS_DIR)

    if raw_docs:
        # 2. Embed & Index (Make sure to run: 'ollama pull nomic-embed-text' first)
        db = create_vector_store(raw_docs, embedding_model_name="nomic-embed-text")

        # 3. Query (Make sure to run: 'ollama pull llama3.2' first)
        run_rag_pipeline(db, model_name="llama3.2")
    else:
        print("No documents found to process.")